# LC 10 — Regular Expression Matching
**Difficulty:** Hard | **Category:** String DP
**Pattern:** 2D DP — Match / Skip with Wildcard

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> Build dp[i][j] = "does s[:i]
match p[:j]". The star '*' is the hard case: it can mean zero
occurrences of the preceding char (skip two pattern chars) OR
one-or-more matches (consume one string char, stay at same
pattern position). Dot '.' matches any single character.
</div>


## Official Problem Statement

Given an input string `s` and a pattern `p`, implement regular
expression matching with support for `.` and `*` where:
- `.` matches any single character.
- `*` matches zero or more of the preceding element.

The matching must cover the entire input string (not partial).

**Constraints:**
- `1 <= s.length <= 20`
- `1 <= p.length <= 30`
- `s` contains only lowercase English letters.
- `p` contains only lowercase English letters, `.`, and `*`.
- It is guaranteed for each occurrence of `*`, there will be
  a previous valid character to match.


## What This Is Actually Asking

Does the entire string `s` match the pattern `p`?
A dot matches any one character. A star means the char before
it can appear zero or more times — 'a*' can match '', 'a',
'aa', 'aaa', etc. You must match the WHOLE string, not just
find a pattern inside it. This is much harder than simple
substring search because of the zero-occurrence case.


## Walk Through an Example by Hand

s = "aa", p = "a*"

Build dp table (rows = s prefix length 0..2,
               cols = p prefix length 0..2):
```
       "" 'a' '*'
   j=   0   1   2
"" i=0  T   F   T   <- dp[0][2]: 'a*' = zero a's = matches ''
'a' i=1  F   T   T   <- dp[1][2]: 'a*' matches 'a'
'a' i=2  F   F   T   <- dp[2][2]: 'a*' matches 'aa'
```

dp[0][2] = T because:
  p[1]='*', so try zero occurrences: dp[0][0] = T

dp[1][2] = T because:
  p[1]='*', zero case: dp[1][0]=F
  one+ case: s[0]='a' matches p[0]='a', so dp[0][2]=T

dp[2][2] = T because:
  p[1]='*', one+ case: s[1]='a' matches p[0]='a', dp[1][2]=T
```
Answer: dp[2][2] = True
```


## The Picture

Full DP table for `s = "aab"`, `p = "c*a*b"`:

```
     "" 'c' '*' 'a' '*' 'b'
j=    0   1   2   3   4   5
""  i=0  T   F   T   F   T   F
'a' i=1  F   F   F   T   T   F
'a' i=2  F   F   F   F   T   F
'b' i=3  F   F   F   F   F   T   <- ANSWER

Recurrence:

def match(si, pj):          # 1-indexed into s, p
    return (p[pj-1] == '.'  # dot matches anything
            or s[si-1] == p[pj-1])  # exact match

if p[j-1] == '*':
    dp[i][j] = dp[i][j-2]           # zero occurrences
             or (match(i, j-1)       # char before * matches
                 and dp[i-1][j])    # consume one from s
else:
    dp[i][j] = match(i, j) and dp[i-1][j-1]
```


## When To Use This Pattern

- When you see regex matching with `.` and `*`, think 2D DP.
- When a character can be used zero or more times, think of
  two sub-cases: skip it (zero) or consume from string (one+).
- When the answer depends on sub-problem "does prefix match
  prefix", think dp[i][j] tables.
- When you see wildcard matching problems (LC 44 uses '?'),
  the same structure applies.
- When constraints are small (s <= 20, p <= 30), full 2D DP
  table is always safe.


## The Approach

Create a 2D boolean table dp of size (len(s)+1) x (len(p)+1).
Base case: dp[0][0] = True (empty matches empty). Handle
patterns like 'a*b*' matching empty string in the first row.
Fill row by row: if pattern char is '*', check zero-occurrence
(look back two pattern cols) and one-or-more (look up one row
if chars match). Otherwise check single char match and
carry dp[i-1][j-1]. Return dp[m][n].


In [ ]:
from typing import List  # standard type hints


In [ ]:
def test_harness(func):
    """Test LC 10 — Regular Expression Matching."""
    tests = [
        # (s, p, expected)
        ("aa",  "a",    False),  # 'a' doesn't match 'aa'
        ("aa",  "a*",   True),   # a* matches 'aa'
        ("ab",  ".*",   True),   # .* matches anything
        ("aab", "c*a*b", True),  # c*=0 c's, a*=2 a's, b
        ("mississippi", "mis*is*p*.", False),
        ("",    "a*",   True),   # zero a's matches empty
        ("a",   ".",    True),   # dot matches single char
        ("a",   "b",    False),  # no match
        ("ab",  "a.",   True),   # a then any char
    ]
    passed = 0
    for s, p, expected in tests:
        result = func(s, p)
        ok = (result == expected)
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        else:
            print(
                f"{status} s={s!r} p={p!r} | "
                f"expected={expected} | got={result}"
            )
    total = len(tests)
    print(f"\n{passed}/{total} tests passed.")


In [ ]:
def isMatch(s: str, p: str) -> bool:
    """
    LC 10 — Regular Expression Matching

    Restatement:
        Return True if s fully matches pattern p.
        '.' matches any single char.
        '*' matches zero or more of the preceding char.

    Approach:
        2D DP. dp[i][j] = s[:i] matches p[:j].
        Base dp[0][0]=True. For '*': try zero (dp[i][j-2])
        or one+ (match char and dp[i-1][j]).
        Else: single char match + dp[i-1][j-1].

    Time:  O(m*n) — m=len(s), n=len(p)
    Space: O(m*n) — dp table
    """
    pass


# Quick debug prints (expected in comments)
print(isMatch("aa",  "a"))       # False
print(isMatch("aa",  "a*"))      # True
print(isMatch("ab",  ".*"))      # True
print(isMatch("aab", "c*a*b"))   # True
print(isMatch("",    "a*"))      # True


In [ ]:
# Uncomment and run when solution is ready
# test_harness(isMatch)


## Complexity

| Approach           | Time    | Space   |
|--------------------|---------|----------|
| Recursive (naive)  | O(2^n)  | O(n)     |
| Memoized recursion | O(m*n)  | O(m*n)   |
| 2D DP (bottom-up)  | O(m*n)  | O(m*n)   |
| 1D DP (optimized)  | O(m*n)  | O(n)     |


## Real World Connection

Regex matching is foundational in Citi's ETL validation layer —
AWS Glue jobs use regex to validate field formats (account IDs,
date strings, transaction codes) before loading to Redshift.
The '.*' pattern in log parsing across 6,000 endpoints catches
any variant of a known error signature. The DP approach mirrors
how NFA-based regex engines work internally: each state
transition is exactly the dp[i][j] recurrence, deciding whether
to consume a character or skip via zero-occurrence star.


> **Simplicity and clarity is Gold.** — Sean's Study Mantra
